In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth agno lancedb sentence-transformers pillow pandas tqdm
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.0
!pip install --no-deps trl==0.22.2

In [5]:
!pip install agno lancedb sentence-transformers pillow pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.2/39.2 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 21.1 MB/s eta 0:00:00


In [14]:
import nest_asyncio
nest_asyncio.apply()  # ✅ MUST be before importing agno


In [10]:
from unsloth import FastVisionModel
from agno.agent import Agent
from agno.db.sqlite import SqliteDb
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.embedder.sentence_transformer import SentenceTransformerEmbedder
from agno.models.openai import OpenAIChat
from agno.vectordb.lancedb import LanceDb
from agno.tools.knowledge import KnowledgeTools
import torch
import pandas as pd
import os
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path
import json
import base64
from io import BytesIO

/tmp/ipython-input-2646005515.py:1: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastVisionModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
ERROR 12-05 10:56:17 [fa_utils.py:72] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


In [63]:
political_kb_content = """
## Sheikh Hasina
Sheikh Hasina Wazed is the former Prime Minister of Bangladesh who served from 2009 to 2024. She is the leader of the Awami League party, whose symbol is a boat (nouka). She is the daughter of Sheikh Mujibur Rahman, the founder of Bangladesh. Common names: Hasina, Sheikh Hasina, PM Hasina. Party colors: red and green.

## Awami League Symbol
The symbol of Awami League is a boat (nouka in Bengali). It appears on a red background and is the party's official election symbol. If you see a boat symbol in a meme, it's likely political and related to Awami League.

## Khaleda Zia
Begum Khaleda Zia is the former Prime Minister of Bangladesh and current chairperson of the Bangladesh Nationalist Party (BNP). She served as PM from 1991-1996 and 2001-2006. The BNP symbol is a sheaf of paddy (rice). She is the widow of President Ziaur Rahman.

## BNP Party Symbol
The BNP (Bangladesh Nationalist Party) symbol is a sheaf of paddy (dhan er sheesh), which represents rice stalks. It appears on a green background. If you see this symbol, the content is political and related to BNP.

## Jamat Shibir/ Hefazat e Islam/ Daripalla
Affiliated political group of BNP

## Joy Bangla Slogan
"Joy Bangla" means "Victory to Bengal" in English. It is the national slogan of Bangladesh and strongly associated with the Awami League party and the 1971 Liberation War. If you see this slogan in a meme, it indicates political content related to Awami League or nationalist sentiment.

## 2024 Bangladesh Protests
In July-August 2024, massive student-led protests occurred in Bangladesh over the quota system in government jobs. These protests led to the resignation of Prime Minister Sheikh Hasina on August 5, 2024. Keywords: quota reform, student movement, August 2024. Any meme referencing these events is highly political.

## Tarique Rahman
Tarique Rahman (born November 20, 1965/1967) is the acting chairman of the Bangladesh Nationalist Party (BNP) and son of former President Ziaur Rahman and former Prime Minister Khaleda Zia. He has been leading the party from exile in London since 2018.

## Nahid Islam
Nahid Islam (born 1998) is the convenor of the National Citizen(s) Party (NCP). He was the coordinator of the 2024 student movement against quotas that evolved into the campaign to oust Prime Minister Sheikh Hasina.

## Obaidul Quader
Obaidul Quader (born February 1, 1952) is the General Secretary of Bangladesh Awami League. He served as Minister of Road Transport and Bridges from 2011–2024.

## Nurul Haque Nur
Nurul Haque Nur (born October 1991), commonly known as VP Nur, is a Bangladeshi activist elected Vice President of DUCSU in 2019.

## Bangladesh Chhatra League (BCL)
Bangladesh Chhatra League (BCL) is the student wing of the Awami League. Look for boat symbol and red-green colors.

## Bangladesh Islami Chhatra Shibir
Bangladesh Islami Chhatra Shibir is the student wing of Jamaat-e-Islami Bangladesh.

## Mamunul Haque
Mamunul Haque (born November 1973) is the Joint Secretary-General of Hefazat-e-Islam Bangladesh and a prominent Islamic scholar.

## National Citizen Party (NCP)
The National Citizen(s) Party (NCP) is a youth and student-led political party formally launched on February 28, 2025, by leaders of the 2024 uprising.

## Nahid Islam
NCP political member

## Rumin Farhana
BNP lawmaker

## Serjis Alam
July Student Protest co leader, NCP member, political figure

## Hasnat Abdullah
July Student Protest co leader, NCP member, political figure

## JD vance
Current US Vice President

## Donald Trump
Current US President

## Zohran Mamdani
Mayorl elect of New York. Muslim. Democratic Socialist

## Ducsu
A student political wing of Dhaka Universisty

## July Revolution 2024
The July Revolution 2024 refers to the student-driven uprising that began with quota reform protests and escalated into a nationwide anti-autocracy movement, culminating in the fall of Sheikh Hasina's government on August 5, 2024.

## Political Classification Indicators
Political content includes: Party symbols (boat, sheaf of paddy), Political leaders, Slogans like "Joy Bangla", Government policies, Student political organizations, Party colors.
"""

# Save knowledge base to file
with open('political_knowledge.md', 'w', encoding='utf-8') as f:
    f.write(political_kb_content)

print("✅ Political knowledge base created: political_knowledge.md")

✅ Political knowledge base created: political_knowledge.md


In [12]:
print("Loading vision model...")
max_seq_length = 16384
lora_rank = 16

base_model, base_tokenizer = FastVisionModel.from_pretrained(
    model_name="arafid/Q3-8B-GRPO-Balanced-Vision-Freezed",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
    gpu_memory_utilization=0.8,
)

FastVisionModel.for_inference(base_model)
print("✅ Vision model loaded successfully")


Loading vision model...
==((====))==  Unsloth 2025.11.6: Fast Qwen3_Vl patching. Transformers: 4.57.0. vLLM: 0.12.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Vision model loaded successfully


In [15]:
!pip install nest_asyncio

In [64]:
print("\nSetting up Agno knowledge base...")

# Create embedder - SentenceTransformer (free, no API key)
embedder = SentenceTransformerEmbedder(
    id="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    # For Bengali support: "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Create vector database - LanceDb
vector_db = LanceDb(
    table_name="political_kb",
    uri="/content/vec_db",
    embedder=embedder,
)

# Create knowledge base - Agno V2 API
knowledge = Knowledge(vector_db=vector_db)

# Add content from markdown file
knowledge.add_content(path="/content/political_knowledge.md")

print("✅ Knowledge base loaded and indexed with Agno")


Setting up Agno knowledge base...


INFO Adding content from path, ff03f755-f289-5fcd-b6ef-37bc868c5512, None, /content/political_knowledge.md, None

WARNING  Contents DB not found for knowledge base

✅ Knowledge base loaded and indexed with Agno


In [46]:
# Verify it worked
search_test = knowledge.search(query="Sheikh Hasina political")
print(f"✅ Knowledge base ready: {len(search_test) if search_test else 0} results found")

INFO Found 3 documents

✅ Knowledge base ready: 3 results found


## Not tested but may use for prompt

In [ ]:
# Cell 6: Create Agno Agent with Vision Model
print("\nCreating Agno agent...")

# Wrap Unsloth model for Agno
vision_model = HuggingFaceVision(
    id="arafid/Q3-8B-GRPO-Balanced-Vision-Freezed",
    model=base_model,
    tokenizer=base_tokenizer,
)

# Create agent with political knowledge
agent = Agent(
    name="PoliticalMemeClassifier",
    model=vision_model,
    knowledge=knowledge,
    description="Expert political meme classifier for Bangladesh politics",
    instructions=[
        "You are an expert in Bangladesh politics and political content classification.",
        "Analyze meme images to determine if they are Political or NonPolitical.",
        "Use the knowledge base to identify political figures, party symbols, slogans, and events.",
        "",
        "**Political** means the meme's PRIMARY content is about:",
        "- Politicians (Sheikh Hasina, Khaleda Zia, Tarique Rahman, Nahid Islam, etc.)",
        "- Political parties (Awami League, BNP, Jamaat, NCP, etc.)",
        "- Party symbols (boat for AL, sheaf of paddy for BNP)",
        "- Political slogans (Joy Bangla, Bangladesh Zindabad)",
        "- Government policies, elections, political campaigns",
        "- Student movements (2024 quota protests, July Revolution)",
        "- Political ideologies, movements, or protests",
        "",
        "**NonPolitical** means the meme is about anything else:",
        "- Gender, relationships, dating",
        "- Religion without political context",
        "- Everyday life, work, school",
        "- Entertainment, movies, games, sports",
        "- Animals, food, technology",
        "- General humor without political context",
        "",
        "IMPORTANT:",
        "1. Search the knowledge base for political figures, symbols, and events",
        "2. Only classify as Political if the MAIN subject is politics",
        "3. If politics is just mentioned briefly, classify as NonPolitical",
        "4. Answer with ONLY: 'Political' or 'NonPolitical'",
        "5. Be decisive - avoid 'unclear' classifications unless absolutely necessary",
    ],
    markdown=True,
    search_knowledge=True,  # Enable automatic knowledge search
    read_chat_history=False,  # Each image is independent
)

print("✅ Agent created successfully")

In [23]:
!pip install vllm

INFO: pip is looking at multiple versions of model-hosting-container-standards to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/lib/python3.12/pathlib.py:404: RuntimeWarning: coroutine 'Knowledge.add_content_async' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [26]:
!pip install qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 21.7 MB/s eta 0:00:00


In [61]:
from qwen_vl_utils import process_vision_info

def classify_image_with_knowledge(image_path: str) -> str:
    """
    Two-pass pipeline:
    1) Use the vision model itself to read text from the meme (OCR-style)
    2) Use extracted text as a query into Agno Knowledge
    3) Combine base political knowledge + OCR-based knowledge into final prompt
    4) Ask the model to output ONLY 'Political' or 'NonPolitical'
    """
    try:
        # ------------------------------------------------------------------
        # PASS 1: OCR / TEXT EXTRACTION FROM IMAGE
        # ------------------------------------------------------------------
        ocr_prompt = """You are an OCR assistant and an image describer.
Read ALL text that appears in this meme image (translate any Bengali into proper English)
and return ONLY the raw text content, without any explanation or extra words. Give a description of the image, highlighting any political party logos.
"""

        ocr_conv = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": Image.open(image_path).convert("RGB")},
                    {"type": "text", "text": ocr_prompt},
                ],
            }
        ]

        # Standard HF multimodal encoding
        ocr_text = base_tokenizer.apply_chat_template(
            ocr_conv, tokenize=False, add_generation_prompt=True
        )
        ocr_images, ocr_videos = process_vision_info(ocr_conv)

        ocr_inputs = base_tokenizer(
            text=[ocr_text],
            images=ocr_images,
            videos=ocr_videos,
            padding=True,
            return_tensors="pt",
        ).to("cuda")

        ocr_ids = base_model.generate(**ocr_inputs, max_new_tokens=128, temperature=0.1)
        ocr_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(ocr_inputs.input_ids, ocr_ids)
        ]
        ocr_response = base_tokenizer.batch_decode(
            ocr_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        extracted_text = ocr_response

        # ------------------------------------------------------------------
        # PASS 2: KNOWLEDGE RETRIEVAL FROM AGNO USING OCR TEXT
        # ------------------------------------------------------------------
        # Base political query plus extracted meme text as extra signal
        kb_query = (
            "Bangladesh political symbols, parties, leaders, slogans, events. "
            f"Meme text: {extracted_text}"
        )

        kb_results = knowledge.search(
            query=kb_query,
            max_results=5,
        )

        kb_context = ""
        if kb_results:
            kb_context = "\n".join([
                f"- {getattr(r, 'content', str(r))[:200]}"
                for r in kb_results[:5]
            ])

        # ------------------------------------------------------------------
        # PASS 3: FINAL CLASSIFICATION PROMPT WITH OCR + KNOWLEDGE
        # ------------------------------------------------------------------
        # Reload image for safety (could reuse from OCR step if you prefer)
        image = Image.open(image_path).convert("RGB")

        print("Extracted Text: ",extracted_text)
        print("KB Context: ", kb_context)

        final_prompt = f"""Classify this meme image as ONLY ONE of: Political or NonPolitical.

TEXT EXTRACTED FROM THE MEME:
{extracted_text}

RETRIEVED POLITICAL KNOWLEDGE (from knowledge base):
{kb_context}

You are an expert in Bangladesh politics and political content classification.
Use BOTH the extracted text and the retrieved knowledge to decide.

Political means the meme's PRIMARY content is about:
- Politicians (Sheikh Hasina, Khaleda Zia, Tarique Rahman, Nahid Islam, etc.)
- Political parties (Awami League, BNP, Jamaat, NCP, etc.)
- Party symbols (boat for AL, sheaf of paddy for BNP)
- Political slogans (Joy Bangla, Bangladesh Zindabad)
- Government policies, elections, political campaigns
- Student movements (2024 quota protests, July Revolution)
- Political ideologies, movements, or protests

NonPolitical means the meme is about anything else:
- Gender, relationships, dating
- Religion without political context
- Everyday life, work, school
- Entertainment, movies, games, sports
- Animals, food, technology
- General humor without political context

IMPORTANT INSTRUCTIONS:
1. Only classify as Political if the MAIN subject is politics.
2. If politics is just mentioned briefly or as background, classify as NonPolitical.
3. Output ONLY ONE WORD, exactly: Political or NonPolitical.
4. Do NOT output explanations, reasoning, or any extra text.
"""

        final_conv = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": final_prompt},
                ],
            }
        ]

        text = base_tokenizer.apply_chat_template(
            final_conv, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(final_conv)

        inputs = base_tokenizer(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to("cuda")

        generated_ids = base_model.generate(
            **inputs,
            max_new_tokens=5,      # Short, just one word
            temperature=0.1,
        )
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        response = base_tokenizer.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        response_upper = response.upper()

        if "POLITICAL" in response_upper and "NON" not in response_upper:
            return "Political"
        elif "NONPOLITICAL" in response_upper or "NON-POLITICAL" in response_upper:
            return "NonPolitical"
        else:
            return "NonPolitical"

    except Exception as e:
        print(f"❌ Error processing {image_path}: {str(e)}")
        return "NonPolitical"


print("✅ Classification function ready (two-pass OCR + Agno Knowledge + Qwen VL)")


✅ Classification function ready (two-pass OCR + Agno Knowledge + Qwen VL)


In [6]:
import zipfile
import os
import pandas as pd

#!unzip /content/Image.zip -d /content/Image
print("\n🔍 Loading test data...")

if os.path.exists("Test.csv"):
    test_df = pd.read_csv("Test.csv")
    print(f"✅ Test.csv loaded: {len(test_df)} images to classify")
else:
    raise FileNotFoundError("Test.csv not found!")

# Find image folder
image_folder ='/content/Image/Image'
for item in os.listdir('.'):
    if os.path.isdir(item) and item not in ['sample_data', '.config', 'tmp']:
        try:
            files = os.listdir(item)
            images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if len(images) > 0:
                image_folder = item
                print(f"✅ Found {len(images)} images in: {image_folder}")
                break
        except:
            continue

if image_folder is None:
    raise FileNotFoundError("No image folder found!")

# Verify images
missing_count = sum(1 for img in test_df['Image_name']
                   if not os.path.exists(os.path.join(image_folder, img)))
if missing_count == 0:
    print(f"✅ All {len(test_df)} test images verified")
else:
    print(f"⚠️ Warning: {missing_count} images not found")


🔍 Loading test data...
✅ Test.csv loaded: 330 images to classify
✅ All 330 test images verified


In [28]:
from tqdm import tqdm
print(f"\n🚀 Starting classification of {len(test_df)} images...\n")

predictions = []
successful = 0
failed = 0

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Classifying"):
    image_name = row['Image_name']
    image_path = os.path.join(image_folder, image_name)

    if not os.path.exists(image_path):
        predictions.append({'Image_name': image_name, 'Label': 'NonPolitical'})
        failed += 1
        continue

    try:
        # Classify using Unsloth + Agno Knowledge
        classification = classify_image_with_knowledge(image_path)
        predictions.append({'Image_name': image_name, 'Label': classification})
        successful += 1
    except Exception as e:
        predictions.append({'Image_name': image_name, 'Label': 'NonPolitical'})
        failed += 1


🚀 Starting classification of 330 images...



Classifying:   0%|          | 0/330 [00:00<?, ?it/s]

INFO Found 5 documents

Classifying:   0%|          | 1/330 [01:35<8:45:06, 95.77s/it]

INFO Found 5 documents

...

Classifying:  99%|█████████▉| 328/330 [28:45<00:06,  3.25s/it]

INFO Found 5 documents

Classifying: 100%|█████████▉| 329/330 [28:53<00:04,  4.89s/it]

INFO Found 5 documents

Classifying: 100%|██████████| 330/330 [28:59<00:00,  5.27s/it]


In [29]:
# Create DataFrame from predictions
results_df = pd.DataFrame(predictions)

# Display summary
print("\n" + "="*80)
print("✅ INFERENCE COMPLETE!")
print("="*80)
print(f"\n📊 Summary:")
print(f"   ✅ Successful: {successful}")
print(f"   ❌ Failed: {failed}")
print(f"   📝 Total: {len(predictions)}")

# Show class distribution
print(f"\n📊 Classification Distribution:")
print(results_df['Label'].value_counts())
print(f"\n📊 Percentages:")
print(results_df['Label'].value_counts(normalize=True) * 100)

# Save to CSV
output_filename = 'rag_predictions.csv'
results_df.to_csv(output_filename, index=False)

print(f"\n✅ Results saved to: {output_filename}")
print(f"\n📋 First 10 predictions:")
print(results_df.head(10))
print(f"\n📋 Last 10 predictions:")
print(results_df.tail(10))


✅ INFERENCE COMPLETE!

📊 Summary:
   ✅ Successful: 330
   ❌ Failed: 0
   📝 Total: 330

📊 Classification Distribution:
Label
NonPolitical    181
Political       149
Name: count, dtype: int64

📊 Percentages:
Label
NonPolitical    54.848485
Political       45.151515
Name: proportion, dtype: float64

✅ Results saved to: rag_predictions.csv

📋 First 10 predictions:
     Image_name         Label
0  test0001.jpg     Political
1  test0002.jpg  NonPolitical
2  test0003.jpg     Political
3  test0004.jpg     Political
4  test0005.jpg  NonPolitical
5  test0006.jpg     Political
6  test0007.jpg  NonPolitical
7  test0008.jpg     Political
8  test0009.jpg  NonPolitical
9  test0010.jpg  NonPolitical

📋 Last 10 predictions:
       Image_name         Label
320  test0321.jpg  NonPolitical
321  test0322.jpg     Political
322  test0323.jpg     Political
323  test0324.jpg     Political
324  test0325.jpg  NonPolitical
325  test0326.jpg  NonPolitical
326  test0327.jpg  NonPolitical
327  test0328.jpg  NonPoli

## Testing with single image

In [67]:
classification = classify_image_with_knowledge('/content/Image/Image/test0024.jpg')
print(classification)

INFO Found 5 documents

Extracted Text:  YES!
YES!
YES!
THE PANGASH
No, God, please, no!
No! No!
জাতীয় নাগরিক পার্টি
এনসিপি
الدين اقیمہو
KB Context:  -  ## Sheikh Hasina Sheikh Hasina Wazed is the former Prime Minister of Bangladesh who served from 2009 to 2024. She is the leader of the Awami League party, whose symbol is a boat (nouka). She is the d
-  ## Sheikh Hasina Sheikh Hasina Wazed is the former Prime Minister of Bangladesh who served from 2009 to 2024. She is the leader of the Awami League party, whose symbol is a boat (nouka). She is the d
-  # Bangladesh Political Figures and Symbols Knowledge Base ## Sheikh Hasina Sheikh Hasina Wazed is the former Prime Minister of Bangladesh who served from 2009 to 2024. She is the leader of the Awami 
-  # Bangladesh Political Figures and Symbols Knowledge Base ## Sheikh Hasina Sheikh Hasina Wazed is the former Prime Minister of Bangladesh who served from 2009 to 2024. She is the leader of the Awami 
-  # Bangladesh Political Figures and Symbols Knowledge Base 